# 02 — DBSCAN Exploration

Unsupervised anomaly signal on its own, before fusing it with the classifier.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

from intelligence.pipeline import config
from intelligence.pipeline.detectors import DBSCANDetector
from intelligence.pipeline.preprocessing import Preprocessor

preprocessor = Preprocessor()
df = preprocessor.scale(preprocessor.clean(preprocessor.load()))
X = df[config.FEATURE_COLUMNS]
X.shape

In [ ]:
detector = DBSCANDetector()
scores = detector.score(X)
noise_rate = scores.mean()
print(f"Flagged as noise (potential fraud): {noise_rate:.2%} of rows")

In [ ]:
if config.LABEL_COLUMN in df.columns:
    import pandas as pd

    comparison = pd.DataFrame({"true_label": df[config.LABEL_COLUMN], "dbscan_noise": scores})
    print(pd.crosstab(comparison["true_label"], comparison["dbscan_noise"]))

In [ ]:
coords = PCA(n_components=2, random_state=config.RANDOM_STATE).fit_transform(X)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(coords[scores == 0, 0], coords[scores == 0, 1], s=4, alpha=0.4, label="clustered (normal)")
ax.scatter(coords[scores == 1, 0], coords[scores == 1, 1], s=8, color="red", label="noise (anomaly)")
ax.legend()
ax.set_title("DBSCAN result in 2D PCA space")
plt.show()